# Compare Backends

Runs the same dataframe and epsilon through every installed SynthHub backend. AIM and MST require Private-PGM mechanisms to be importable.

In [ ]:
# Install SynthHub from PyPI when running in Colab or a clean notebook.
%pip install -q "synthhub[datasynthesizer]"


In [ ]:
import pandas as pd
from synthhub import Synthesizer
from synthhub.errors import SynthHubError

real_df = pd.DataFrame({
    "age": [21, 34, 37, 45, 52, 23, 41, 29, 62, 31] * 20,
    "city": ["A", "B", "A", "C", "B", "A", "C", "C", "B", "A"] * 20,
    "churn": [0, 1, 0, 1, 1, 0, 1, 0, 1, 0] * 20,
})

preferred_order = [
    "aim", "mst", "privbayes", "datasynthesizer-privbayes", "datasynthesizer-independent", "independent",
    "mwem", "pacsynth", "dpctgan", "patectgan", "pategan", "dpgan", "quail",
    "smartnoise-aim", "smartnoise-mst",
    "synthcity-privbayes", "synthcity-pategan", "synthcity-dpgan",
]
methods = [method for method in preferred_order if method in Synthesizer.available_methods()]

rows = []
for method in methods:
    try:
        synth = Synthesizer(method=method, epsilon=1.0, random_state=0)
        synth.fit(real_df)
        synthetic_df = synth.sample(len(real_df))
        report = synth.evaluate(real_df, synthetic_df, target="churn").to_dict()
        rows.append({
            "method": method,
            "epsilon_spent": report["privacy"]["accounting"]["epsilon_spent"],
            "utility_score": report["utility"]["mean_column_similarity"],
            "reid_risk": report["privacy"]["membership_inference"].get("risk_score"),
            "status": "ok",
        })
    except Exception as exc:
        rows.append({"method": method, "status": type(exc).__name__, "detail": str(exc)})

pd.DataFrame(rows)